In [1]:
import os
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from docxtpl import DocxTemplate
from datetime import datetime

# --- 1. CONFIGURACIÓN DE RUTAS Y DATOS ---
ruta_base = r"G:\Ingenio Azucarero Guabira S.A\UTEA - SEMANAL - PROGRAMA DE COSECHA\2026\SEGUIMIENTO_COSECHA"
ruta_plantilla = r"G:\OneDrive - Ingenio Azucarero Guabira S.A\_DATOS_PYTHON\templates"

# Carga de ambas bases de datos
df_ha = pd.read_csv(os.path.join(ruta_base, "REPORTE_SERVICIO_HA_COSECHADA.csv"), encoding="utf-8")
df_detalles = pd.read_csv(os.path.join(ruta_base, "REPORTE_SERVICIO_DETALLE_LOTES.csv"), encoding="utf-8")

meses_espanol = {
    1: 'enero', 2: 'febrero', 3: 'marzo', 4: 'abril', 5: 'mayo', 6: 'junio',
    7: 'julio', 8: 'agosto', 9: 'septiembre', 10: 'octubre', 11: 'noviembre', 12: 'diciembre'
}

# --- 2. FUNCIÓN DE RENDERIZADO CON NUEVOS CAMPOS (VARIEDAD Y SOCA) ---
def generar_reporte_servicio_completo(propiedad, fecha_control):
    try:
        plantilla_path = os.path.join(ruta_plantilla, "tpl_inf_cosecha_servicio_F1.docx")
        doc = DocxTemplate(plantilla_path)
        
        anio_sel = fecha_control.year
        mes_nombre_sel = meses_espanol[fecha_control.month]
        semana_sel = fecha_control.isocalendar()[1]
        
        # --- BLOQUE 1: CÁLCULOS MATEMÁTICOS (df_ha) ---
        df_prop = df_ha[(df_ha['Propiedad'] == propiedad) & (df_ha['Año'] == anio_sel)]
        
        # Filtro Semanal
        df_sem = df_prop[(df_prop['Mes'].str.lower() == mes_nombre_sel) & (df_prop['Semana'] == semana_sel)]
        sem_area = df_sem['Area Cosechada'].sum()
        sem_tn = df_sem['Tn Real'].sum()
        sem_tch = (sem_tn / sem_area) if sem_area > 0 else 0.0
        
        # Filtro Mensual
        df_mes = df_prop[df_prop['Mes'].str.lower() == mes_nombre_sel]
        mes_area = df_mes['Area Cosechada'].sum()
        mes_tn = df_mes['Tn Real'].sum()
        mes_tch = (mes_tn / mes_area) if mes_area > 0 else 0.0
        
        # Acumulado Total
        total_area = df_prop['Area Cosechada'].sum()
        total_tn = df_prop['Tn Real'].sum()
        total_tch = (total_tn / total_area) if total_area > 0 else 0.0
        
        # --- BLOQUE 2: PROCESAMIENTO Y LIMPIEZA DE LA TABLA DE LOTES ---
        df_lotes_prop = df_detalles[df_detalles['Propiedad'] == propiedad].copy()
        
        # Rellenar nulos numéricos por seguridad
        df_lotes_prop['Area Cosecha'] = df_lotes_prop['Area Cosecha'].fillna(0.0)
        df_lotes_prop['Tn Real'] = df_lotes_prop['Tn Real'].fillna(0.0)
        df_lotes_prop['TCH Real'] = df_lotes_prop['TCH Real'].fillna(0.0)
        df_lotes_prop['Variedad'] = df_lotes_prop['Variedad'].fillna("S/V")  # Sin Variedad
        df_lotes_prop['Soca'] = df_lotes_prop['Soca'].fillna(0)
        
        lista_lotes_jinja = []
        for _, fila in df_lotes_prop.iterrows():
            
            # Lógica de limpieza para el Avance (Sin decimales y tope 100%)
            avance_original = fila['Avance']
            if pd.isna(avance_original):
                avance_final = "0 %"
            else:
                try:
                    avance_limpio = str(avance_original).replace('%', '').replace(',', '.').strip()
                    avance_num = float(avance_limpio)
                    if avance_num > 100.0:
                        avance_num = 100.0
                    avance_final = f"{int(round(avance_num))} %"
                except:
                    avance_final = str(avance_original)
            
            # Inyección de datos al diccionario de Jinja
            lista_lotes_jinja.append({
                "lote": str(fila['Lote']),
                "estado": str(fila['Estado Cosecha']),
                "avance": avance_final,
                "variedad": str(fila['Variedad']),      # NUEVO
                "soca": str(int(fila['Soca'])),         # NUEVO (convertido a entero)
                "area": f"{fila['Area Cosecha']:,.2f}",
                "tn": f"{fila['Tn Real']:,.2f}",
                "tch": f"{fila['TCH Real']:,.2f}"
            })
            
        # --- BLOQUE 3: CONTEXTO INTEGRADO PARA JINJA ---
        contexto_r = {
            "fecha": fecha_control.strftime("%d/%m/%Y"),
            "propiedad": propiedad,
            "num_semana": semana_sel,
            "nombre_mes": mes_nombre_sel.capitalize(),
            
            # Resúmenes de Hectáreas
            "sem_area": f"{sem_area:,.2f}", "sem_tn": f"{sem_tn:,.2f}", "sem_tch": f"{sem_tch:,.2f}",
            "mes_area": f"{mes_area:,.2f}", "mes_tn": f"{mes_tn:,.2f}", "mes_tch": f"{mes_tch:,.2f}",
            "total_area": f"{total_area:,.2f}", "total_tn": f"{total_tn:,.2f}", "total_tch": f"{total_tch:,.2f}",
            
            "lotes": lista_lotes_jinja
        }
        
        # Renderizar y Guardar
        doc.render({"r": contexto_r})
        nombre_archivo = f"Reporte_Servicio_Completo_{propiedad.replace(' ', '_')}_Semana_{semana_sel}.docx"
        ruta_salida = os.path.join(ruta_base, nombre_archivo)
        doc.save(ruta_salida)
        
        return f"✅ ¡Reporte generado con éxito!\n📂 Archivo con Variedad, Soca y Avances corregidos: {ruta_salida}"
        
    except Exception as e:
        return f"❌ Error al procesar el reporte: {str(e)}"

# --- 4. INTERFAZ GRÁFICA INTERACTIVA ---
combo_propiedad = widgets.Dropdown(options=sorted(df_ha['Propiedad'].unique()), description='Propiedad:')
control_fecha = widgets.DatePicker(description='Fecha Consulta:', value=datetime.now().date())
btn_generar = widgets.Button(description='Generar Reporte con Tabla', button_style='success', icon='table')
output_data = widgets.Output()

def controlar_interfaz(*args):
    with output_data:
        clear_output()
        if combo_propiedad.value and control_fecha.value:
            f_sel = control_fecha.value
            print(f"Propiedad: {combo_propiedad.value} | Fecha seleccionada: {f_sel.strftime('%d/%m/%Y')}")
            cant_lotes = len(df_detalles[df_detalles['Propiedad'] == combo_propiedad.value])
            print(f"🚜 Se exportarán {cant_lotes} lotes incluyendo los campos de Variedad y Soca.")
            display(btn_generar)

def al_hacer_clic_generar(b):
    with output_data:
        print("\nConsolidando datos y aplicando reglas de negocio a la columna Avance...")
        # LÍNEA CORREGIDA: Sincronizada exactamente con el nombre de la función principal
        mensaje = generar_reporte_servicio_completo(combo_propiedad.value, control_fecha.value)
        print(mensaje)

combo_propiedad.observe(controlar_interfaz, 'value')
control_fecha.observe(controlar_interfaz, 'value')
btn_generar.on_click(al_hacer_clic_generar)

controlar_interfaz()
display(widgets.HBox([combo_propiedad, control_fecha]), output_data)

Output()

In [18]:
import os
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from docxtpl import DocxTemplate
from datetime import datetime

# --- 1. CONFIGURACIÓN DE RUTAS Y DATOS ---
ruta_base = r"G:\Ingenio Azucarero Guabira S.A\UTEA - SEMANAL - PROGRAMA DE COSECHA\2026\SEGUIMIENTO_COSECHA"
ruta_plantilla = r"G:\OneDrive - Ingenio Azucarero Guabira S.A\_DATOS_PYTHON\templates"

# Carga de ambas bases de datos
df_ha = pd.read_csv(os.path.join(ruta_base, "REPORTE_SERVICIO_HA_COSECHADA.csv"), encoding="utf-8")
df_detalles = pd.read_csv(os.path.join(ruta_base, "REPORTE_SERVICIO_DETALLE_LOTES.csv"), encoding="utf-8")

meses_espanol = {
    1: 'enero', 2: 'febrero', 3: 'marzo', 4: 'abril', 5: 'mayo', 6: 'junio',
    7: 'julio', 8: 'agosto', 9: 'septiembre', 10: 'octubre', 11: 'noviembre', 12: 'diciembre'
}

# --- 2. FUNCIÓN DE RENDERIZADO CON TABLA DE AVANCE Y TABLA DE CALIDAD ---
def generar_reporte_servicio_completo(propiedad, fecha_control):
    try:
        plantilla_path = os.path.join(ruta_plantilla, "tpl_inf_cosecha_servicio_F1.docx")
        doc = DocxTemplate(plantilla_path)
        
        anio_sel = fecha_control.year
        mes_nombre_sel = meses_espanol[fecha_control.month]
        semana_sel = fecha_control.isocalendar()[1]
        
        # --- BLOQUE 1: CÁLCULOS MATEMÁTICOS (df_ha) ---
        df_prop = df_ha[(df_ha['Propiedad'] == propiedad) & (df_ha['Año'] == anio_sel)]
        
        # Filtro Semanal
        df_sem = df_prop[(df_prop['Mes'].str.lower() == mes_nombre_sel) & (df_prop['Semana'] == semana_sel)]
        sem_area = df_sem['Area Cosechada'].sum()
        sem_tn = df_sem['Tn Real'].sum()
        sem_tch = (sem_tn / sem_area) if sem_area > 0 else 0.0
        
        # Filtro Mensual
        df_mes = df_prop[df_prop['Mes'].str.lower() == mes_nombre_sel]
        mes_area = df_mes['Area Cosechada'].sum()
        mes_tn = df_mes['Tn Real'].sum()
        mes_tch = (mes_tn / mes_area) if mes_area > 0 else 0.0
        
        # Acumulado Total
        total_area = df_prop['Area Cosechada'].sum()
        total_tn = df_prop['Tn Real'].sum()
        total_tch = (total_tn / total_area) if total_area > 0 else 0.0
        
        # --- BLOQUE 2: PROCESAMIENTO Y LIMPIEZA DE LA TABLA DE LOTES ---
        df_lotes_prop = df_detalles[df_detalles['Propiedad'] == propiedad].copy()
        
        # Rellenar nulos de la base de datos por seguridad
        df_lotes_prop['Area Cosecha'] = df_lotes_prop['Area Cosecha'].fillna(0.0)
        df_lotes_prop['Tn Real'] = df_lotes_prop['Tn Real'].fillna(0.0)
        df_lotes_prop['TCH Real'] = df_lotes_prop['TCH Real'].fillna(0.0)
        df_lotes_prop['Variedad'] = df_lotes_prop['Variedad'].fillna("S/V")
        df_lotes_prop['Soca'] = df_lotes_prop['Soca'].fillna(0)
        
        # NUEVO: Rellenar nulos para los campos de laboratorio (Ponderados)
        df_lotes_prop['POND ME'] = df_lotes_prop['POND ME'].fillna(0.0)
        df_lotes_prop['POND PCF'] = df_lotes_prop['POND PCF'].fillna(0.0)
        df_lotes_prop['POND FIBRA'] = df_lotes_prop['POND FIBRA'].fillna(0.0)
        df_lotes_prop['POND PUREZA'] = df_lotes_prop['POND PUREZA'].fillna(0.0)
        
        lista_lotes_jinja = []
        for _, fila in df_lotes_prop.iterrows():
            
            # Lógica de limpieza para el Avance (Sin decimales y tope 100%)
            avance_original = fila['Avance']
            if pd.isna(avance_original):
                avance_final = "0 %"
            else:
                try:
                    avance_limpio = str(avance_original).replace('%', '').replace(',', '.').strip()
                    avance_num = float(avance_limpio)
                    if avance_num > 100.0:
                        avance_num = 100.0
                    avance_final = f"{int(round(avance_num))} %"
                except:
                    avance_final = str(avance_original)
            
            # Inyección de datos consolidados de cosecha y laboratorio
            lista_lotes_jinja.append({
                "lote": str(fila['Lote']),
                "estado": str(fila['Estado Cosecha']),
                "avance": avance_final,
                "variedad": str(fila['Variedad']),
                "soca": str(int(fila['Soca'])),
                "area": f"{fila['Area Cosecha']:,.2f}",
                "tn": f"{fila['Tn Real']:,.2f}",
                "tch": f"{fila['TCH Real']:,.2f}",
                
                # NUEVOS INDICADORES DE LABORATORIO (PONDERADOS)
                "me": f"{fila['POND ME']:,.2f} %",
                "pcf": f"{fila['POND PCF']:,.2f}",
                "fibra": f"{fila['POND FIBRA']:,.2f} %",
                "pureza": f"{fila['POND PUREZA']:,.2f} %"
            })
            
        # --- BLOQUE 3: CONTEXTO INTEGRADO PARA JINJA ---
        contexto_r = {
            "fecha": fecha_control.strftime("%d/%m/%Y"),
            "propiedad": propiedad,
            "num_semana": semana_sel,
            "nombre_mes": mes_nombre_sel.capitalize(),
            
            # Resúmenes de Hectáreas
            "sem_area": f"{sem_area:,.2f}", "sem_tn": f"{sem_tn:,.2f}", "sem_tch": f"{sem_tch:,.2f}",
            "mes_area": f"{mes_area:,.2f}", "mes_tn": f"{mes_tn:,.2f}", "mes_tch": f"{mes_tch:,.2f}",
            "total_area": f"{total_area:,.2f}", "total_tn": f"{total_tn:,.2f}", "total_tch": f"{total_tch:,.2f}",
            
            "lotes": lista_lotes_jinja
        }
        
        # Renderizar y Guardar documento
        doc.render({"r": contexto_r})
        nombre_archivo = f"Reporte_Servicio_Completo_{propiedad.replace(' ', '_')}_Semana_{semana_sel}.docx"
        ruta_salida = os.path.join(ruta_base, nombre_archivo)
        doc.save(ruta_salida)
        
        return f"✅ ¡Reporte generado con éxito!\n📂 Guardado en: {ruta_salida}\n📊 Se estructuraron las tablas de Avance Técnico y Calidad de Laboratorio."
        
    except Exception as e:
        return f"❌ Error al procesar el reporte: {str(e)}"

# --- 4. INTERFAZ GRÁFICA INTERACTIVA ---
combo_propiedad = widgets.Dropdown(options=sorted(df_ha['Propiedad'].unique()), description='Propiedad:')
control_fecha = widgets.DatePicker(description='Fecha Consulta:', value=datetime.now().date())
btn_generar = widgets.Button(description='Generar Ambas Tablas', button_style='success', icon='table')
output_data = widgets.Output()

def controlar_interfaz(*args):
    with output_data:
        clear_output()
        if combo_propiedad.value and control_fecha.value:
            f_sel = control_fecha.value
            print(f"Propiedad: {combo_propiedad.value} | Fecha seleccionada: {f_sel.strftime('%d/%m/%Y')}")
            cant_lotes = len(df_detalles[df_detalles['Propiedad'] == combo_propiedad.value])
            print(f"🌾 Se estructurarán {cant_lotes} lotes automáticamente distribuidos en la Tabla Operativa y en la Tabla de Calidad (Ponderados).")
            display(btn_generar)

def al_hacer_clic_generar(b):
    with output_data:
        print("\nConsolidando matrices y dando formato a los ponderados de laboratorio...")
        mensaje = generar_reporte_servicio_completo(combo_propiedad.value, control_fecha.value)
        print(mensaje)

combo_propiedad.observe(controlar_interfaz, 'value')
control_fecha.observe(controlar_interfaz, 'value')
btn_generar.on_click(al_hacer_clic_generar)

controlar_interfaz()
display(widgets.HBox([combo_propiedad, control_fecha]), output_data)

Output()

In [28]:
import os
import sys
sys.path.append('..')
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from sqlalchemy import create_engine, text
from docxtpl import DocxTemplate, InlineImage
from docx.shared import Inches
from datetime import datetime

# --- 1. CONFIGURACIÓN DE RUTAS Y BASE DE DATOS ---
ruta_base = r"G:\Ingenio Azucarero Guabira S.A\UTEA - SEMANAL - PROGRAMA DE COSECHA\2026\SEGUIMIENTO_COSECHA"
ruta_plantilla = r"G:\OneDrive - Ingenio Azucarero Guabira S.A\_DATOS_PYTHON\templates"

from config import POSTGRES_UTEA
POSTGRES_UTEA['DATABASE'] = 'utea_precision'

def obtener_engine():
    return create_engine(
        f"postgresql+psycopg2://{POSTGRES_UTEA['USER']}:{POSTGRES_UTEA['PASSWORD']}@{POSTGRES_UTEA['HOST']}:{POSTGRES_UTEA['PORT']}/{POSTGRES_UTEA['DATABASE']}"
    )

# Carga de ambas bases de datos CSV
df_ha = pd.read_csv(os.path.join(ruta_base, "REPORTE_SERVICIO_HA_COSECHADA.csv"), encoding="utf-8")
df_detalles = pd.read_csv(os.path.join(ruta_base, "REPORTE_SERVICIO_DETALLE_LOTES.csv"), encoding="utf-8")

meses_espanol = {
    1: 'enero', 2: 'febrero', 3: 'marzo', 4: 'abril', 5: 'mayo', 6: 'junio',
    7: 'julio', 8: 'agosto', 9: 'septiembre', 10: 'octubre', 11: 'noviembre', 12: 'diciembre'
}

# Paleta de colores actualizada: Rojo, Naranja, Amarillo, Verde Claro y Verde Intenso
colores_tch = {
    '< 40 Tn/Ha': '#d7191c',       # Rojo Intenso (Rendimiento muy bajo)
    '40 a 60 Tn/Ha': '#fdae61',    # Naranja (Rendimiento regular)
    '60 a 80 Tn/Ha': '#ffffbf',    # Amarillo (Rendimiento bueno / promedio)
    '80 a 100 Tn/Ha': '#a6d96a',   # Verde Claro (Rendimiento muy bueno)
    '> 100 Tn/Ha': '#1a9641'       # Verde Intenso (Rendimiento óptimo / sobresaliente)
}
# --- 2. FUNCIÓN PARA EXTRACTO GEOGRÁFICO DE LA PROPIEDAD COMPLETA ---
def obtener_telemetria_propiedad(propiedad):
    """Descarga los recorridos de todos los lotes pertenecientes a una misma propiedad."""
    engine = obtener_engine()
    query = text("""
        SELECT id, vryldrcane, distance, distancia_real, geom 
        FROM datos_cosecha.recorrido_cosechadora 
        WHERE propiedad = :prop
    """)
    try:
        gdf = gpd.read_postgis(query, engine, geom_col='geom', params={"prop": propiedad})
        return gdf
    except Exception as e:
        print(f"⚠️ Nota: No se pudo extraer mapa desde Postgres para esta propiedad: {e}")
        return gpd.GeoDataFrame()

def generar_plano_tch_general(gdf, path_img):
    """Genera el plano general aplicando el filtro de distancia y la nueva categorización de TCH."""
    gdf = gdf.copy()
    
    # NUEVA CATEGORIZACIÓN EXTENDIDA
    gdf['categoria'] = pd.cut(
        gdf['vryldrcane'], 
        bins=[-1, 40, 60, 80, 100, 9999], # Agregado el límite de 100
        labels=['< 40 Tn/Ha', '40 a 60 Tn/Ha', '60 a 80 Tn/Ha', '80 a 100 Tn/Ha', '> 100 Tn/Ha'], 
        include_lowest=True
    )
    
    # Filtro de limpieza espacial (< 3 metros)
    col_filtro = 'distancia_real' if 'distancia_real' in gdf.columns else 'distance'
    gdf_plano = gdf[gdf[col_filtro] < 3].copy()
    
    fig, ax = plt.subplots(figsize=(10, 5))
    minx, miny, maxx, maxy = gdf_plano.total_bounds
    cx, cy = (minx + maxx) / 2, (miny + maxy) / 2
    rango = max(maxx - minx, (maxy - miny) * 2.5) * 1.2
    
    # Renderizar tramos por categoría de color con el grosor definido (linewidth)
    for cat, data in gdf_plano.groupby('categoria', observed=False):
        data.plot(ax=ax, color=colores_tch.get(cat, '#7f8c8d'), linewidth=0.1, label=str(cat))
        
    ax.set_xlim(cx - rango / 2, cx + rango / 2)
    ax.set_ylim(cy - (rango / 2.5) / 2, cy + (rango / 2.5) / 2)
    ax.axis('off')
    plt.title(f"PLANO GENERAL DE RENDIMIENTO TCH - PROPIEDAD", fontsize=12, fontweight='bold', pad=12)
    plt.legend(loc='lower right', frameon=True)
    plt.savefig(path_img, dpi=100, bbox_inches='tight')
    plt.close()

# --- 3. FUNCIÓN DE RENDERIZADO PRINCIPAL ---
def generar_reporte_servicio_completo(propiedad, fecha_control):
    try:
        plantilla_path = os.path.join(ruta_plantilla, "tpl_inf_cosecha_servicio_F1.docx")
        doc = DocxTemplate(plantilla_path)
        
        anio_sel = fecha_control.year
        mes_nombre_sel = meses_espanol[fecha_control.month]
        semana_sel = fecha_control.isocalendar()[1]
        
        # --- BLOQUE 1: CÁLCULOS MATEMÁTICOS (df_ha) ---
        df_prop = df_ha[(df_ha['Propiedad'] == propiedad) & (df_ha['Año'] == anio_sel)]
        sem_area = df_prop[(df_prop['Mes'].str.lower() == mes_nombre_sel) & (df_prop['Semana'] == semana_sel)]['Area Cosechada'].sum()
        sem_tn = df_prop[(df_prop['Mes'].str.lower() == mes_nombre_sel) & (df_prop['Semana'] == semana_sel)]['Tn Real'].sum()
        sem_tch = (sem_tn / sem_area) if sem_area > 0 else 0.0
        
        mes_area = df_prop[df_prop['Mes'].str.lower() == mes_nombre_sel]['Area Cosechada'].sum()
        mes_tn = df_prop[df_prop['Mes'].str.lower() == mes_nombre_sel]['Tn Real'].sum()
        mes_tch = (mes_tn / mes_area) if mes_area > 0 else 0.0
        
        total_area = df_prop['Area Cosechada'].sum()
        total_tn = df_prop['Tn Real'].sum()
        total_tch = (total_tn / total_area) if total_area > 0 else 0.0
        
        # --- BLOQUE 2: PROCESAMIENTO DE LAS DOS TABLAS DE LOTES ---
        df_lotes_prop = df_detalles[df_detalles['Propiedad'] == propiedad].copy()
        
        df_lotes_prop['Area Cosecha'] = df_lotes_prop['Area Cosecha'].fillna(0.0)
        df_lotes_prop['Tn Real'] = df_lotes_prop['Tn Real'].fillna(0.0)
        df_lotes_prop['TCH Real'] = df_lotes_prop['TCH Real'].fillna(0.0)
        df_lotes_prop['Variedad'] = df_lotes_prop['Variedad'].fillna("S/V")
        df_lotes_prop['Soca'] = df_lotes_prop['Soca'].fillna(0)
        df_lotes_prop['POND ME'] = df_lotes_prop['POND ME'].fillna(0.0)
        df_lotes_prop['POND PCF'] = df_lotes_prop['POND PCF'].fillna(0.0)
        df_lotes_prop['POND FIBRA'] = df_lotes_prop['POND FIBRA'].fillna(0.0)
        df_lotes_prop['POND PUREZA'] = df_lotes_prop['POND PUREZA'].fillna(0.0)
        
        lista_lotes_jinja = []
        for _, fila in df_lotes_prop.iterrows():
            avance_original = fila['Avance']
            if pd.isna(avance_original):
                avance_final = "0 %"
            else:
                try:
                    avance_limpio = str(avance_original).replace('%', '').replace(',', '.').strip()
                    avance_num = float(avance_limpio)
                    if avance_num > 100.0: avance_num = 100.0
                    avance_final = f"{int(round(avance_num))} %"
                except:
                    avance_final = str(avance_original)
            
            lista_lotes_jinja.append({
                "lote": str(fila['Lote']), "estado": str(fila['Estado Cosecha']), "avance": avance_final,
                "variedad": str(fila['Variedad']), "soca": str(int(fila['Soca'])),
                "area": f"{fila['Area Cosecha']:,.2f}", "tn": f"{fila['Tn Real']:,.2f}", "tch": f"{fila['TCH Real']:,.2f}",
                "me": f"{fila['POND ME']:,.2f} %", "pcf": f"{fila['POND PCF']:,.2f}",
                "fibra": f"{fila['POND FIBRA']:,.2f} %", "pureza": f"{fila['POND PUREZA']:,.2f} %"
            })
            
        # --- BLOQUE 3: CONSTRUCCIÓN DEL PLANO MAPA GENERAL (PostgreSQL) ---
        img_temp_mapa = "tmp_tch_general.png"
        gdf_geo = obtener_telemetria_propiedad(propiedad)
        tiene_geo = not gdf_geo.empty
        
        if tiene_geo:
            generar_plano_tch_general(gdf_geo, img_temp_mapa)
            
        # --- BLOQUE 4: CONTEXTO INTEGRADO PARA JINJA ---
        contexto_r = {
            "fecha": fecha_control.strftime("%d/%m/%Y"),
            "propiedad": propiedad,
            "num_semana": semana_sel,
            "nombre_mes": mes_nombre_sel.capitalize(),
            
            "sem_area": f"{sem_area:,.2f}", "sem_tn": f"{sem_tn:,.2f}", "sem_tch": f"{sem_tch:,.2f}",
            "mes_area": f"{mes_area:,.2f}", "mes_tn": f"{mes_tn:,.2f}", "mes_tch": f"{mes_tch:,.2f}",
            "total_area": f"{total_area:,.2f}", "total_tn": f"{total_tn:,.2f}", "total_tch": f"{total_tch:,.2f}",
            
            "lotes": lista_lotes_jinja,
            
            # Inyección de Imagen del Plano consolidado de la Propiedad completa
            "img_plano_tch_general": InlineImage(doc, img_temp_mapa, width=Inches(6.0)) if tiene_geo else "[Propiedad sin telemetría geoespacial registrada]"
        }
        
        doc.render({"r": contexto_r})
        nombre_archivo = f"Reporte_Servicio_Completo_{propiedad.replace(' ', '_')}_Semana_{semana_sel}.docx"
        ruta_salida = os.path.join(ruta_base, nombre_archivo)
        doc.save(ruta_salida)
        
        # Limpieza de archivo temporal
        if os.path.exists(img_temp_mapa): os.remove(img_temp_mapa)
        
        return f"✅ ¡Reporte consolidado con éxito!\n📂 Archivo: {ruta_salida}\n🌍 Incluye: Tablas operativas, de calidad y plano unificado de TCH de la propiedad."
        
    except Exception as e:
        return f"❌ Error al procesar el reporte: {str(e)}"

# --- 5. INTERFAZ GRÁFICA INTERACTIVA ---
combo_propiedad = widgets.Dropdown(options=sorted(df_ha['Propiedad'].unique()), description='Propiedad:')
control_fecha = widgets.DatePicker(description='Fecha Consulta:', value=datetime.now().date())
btn_generar = widgets.Button(description='Generar Reporte con Mapa', button_style='success', icon='map')
output_data = widgets.Output()

def controlar_interfaz(*args):
    with output_data:
        clear_output()
        if combo_propiedad.value and control_fecha.value:
            print(f"Propiedad: {combo_propiedad.value} | Fecha seleccionada: {control_fecha.value.strftime('%d/%m/%Y')}")
            cant_lotes = len(df_detalles[df_detalles['Propiedad'] == combo_propiedad.value])
            print(f"📋 Se unificarán {cant_lotes} lotes de la propiedad para renderizar el Plano General de TCH.")
            display(btn_generar)

def al_hacer_clic_generar(b):
    with output_data:
        print("\n📥 Conectando a PostGIS, combinando geometrías vectoriales y procesando Jinja...")
        mensaje = generar_reporte_servicio_completo(combo_propiedad.value, control_fecha.value)
        print(mensaje)

combo_propiedad.observe(controlar_interfaz, 'value')
control_fecha.observe(controlar_interfaz, 'value')
btn_generar.on_click(al_hacer_clic_generar)

controlar_interfaz()
display(widgets.HBox([combo_propiedad, control_fecha]), output_data)

Output()

In [3]:
import os
import sys
sys.path.append('..')
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import dataframe_image as dfi
import ipywidgets as widgets
from IPython.display import display, clear_output
from sqlalchemy import create_engine, text
from docxtpl import DocxTemplate, InlineImage
from docx.shared import Inches
from datetime import datetime

# --- 1. CONFIGURACIÓN DE RUTAS Y BASE DE DATOS ---
ruta_base = r"G:\Ingenio Azucarero Guabira S.A\UTEA - SEMANAL - PROGRAMA DE COSECHA\2026\SEGUIMIENTO_COSECHA"
ruta_plantilla = r"G:\OneDrive - Ingenio Azucarero Guabira S.A\_DATOS_PYTHON\templates"

from config import POSTGRES_UTEA
POSTGRES_UTEA['DATABASE'] = 'utea_precision'

def obtener_engine():
    return create_engine(
        f"postgresql+psycopg2://{POSTGRES_UTEA['USER']}:{POSTGRES_UTEA['PASSWORD']}@{POSTGRES_UTEA['HOST']}:{POSTGRES_UTEA['PORT']}/{POSTGRES_UTEA['DATABASE']}"
    )

# Carga de ambas bases de datos CSV
df_ha = pd.read_csv(os.path.join(ruta_base, "REPORTE_SERVICIO_HA_COSECHADA.csv"), encoding="utf-8")
df_detalles = pd.read_csv(os.path.join(ruta_base, "REPORTE_SERVICIO_DETALLE_LOTES.csv"), encoding="utf-8")

# =====================================================================
# 🔥 ¡AÑADE ESTAS LÍNEAS AQUÍ ABAJO PARA CORREGIR EL ERROR!
# =====================================================================
# Aseguramos que las columnas de cálculos de df_ha sean números limpios
for col in ['Area Cosechada', 'Tn Real']:
    if col in df_ha.columns:
        # Reemplaza comas por puntos (si vienen en formato latino) y convierte a float
        df_ha[col] = df_ha[col].astype(str).str.replace(',', '.').str.strip()
        df_ha[col] = pd.to_numeric(df_ha[col], errors='coerce').fillna(0.0)

# Aseguramos que las columnas de df_detalles sean números limpios
for col in ['Area Cosecha', 'Tn Real', 'TCH Real', 'POND ME', 'POND PCF', 'POND FIBRA', 'POND PUREZA']:
    if col in df_detalles.columns:
        df_detalles[col] = df_detalles[col].astype(str).str.replace(',', '.').str.strip()
        df_detalles[col] = pd.to_numeric(df_detalles[col], errors='coerce').fillna(0.0)
# =====================================================================

meses_espanol = {
    1: 'enero', 2: 'febrero', 3: 'marzo', 4: 'abril', 5: 'mayo', 6: 'junio',
    7: 'julio', 8: 'agosto', 9: 'septiembre', 10: 'octubre', 11: 'noviembre', 12: 'diciembre'
}

# Paleta de colores actualizada: Rojo, Naranja, Amarillo, Verde Claro y Verde Intenso
colores_tch = {
    '< 40 Tn/Ha': '#d7191c',       # Rojo Intenso (Rendimiento muy bajo)
    '40 a 60 Tn/Ha': '#fdae61',    # Naranja (Rendimiento regular)
    '60 a 80 Tn/Ha': '#ffffbf',    # Amarillo (Rendimiento bueno / promedio)
    '80 a 100 Tn/Ha': '#a6d96a',   # Verde Claro (Rendimiento muy bueno)
    '> 100 Tn/Ha': '#1a9641'       # Verde Intenso (Rendimiento óptimo / sobresaliente)
}

# --- 2. FUNCIÓN PARA EXTRACTO GEOGRÁFICO DE LA PROPIEDAD COMPLETA ---
def obtener_telemetria_propiedad(propiedad):
    """Descarga los recorridos de todos los lotes pertenecientes a una misma propiedad."""
    engine = obtener_engine()
    query = text("""
        SELECT id, vryldrcane, distance, swathwidth, distancia_real, geom 
        FROM datos_cosecha.recorrido_cosechadora 
        WHERE propiedad = :prop
    """)
    try:
        gdf = gpd.read_postgis(query, engine, geom_col='geom', params={"prop": propiedad})
        return gdf
    except Exception as e:
        print(f"⚠️ Nota: No se pudo extraer mapa desde Postgres para esta propiedad: {e}")
        return gpd.GeoDataFrame()

def generar_plano_y_tabla_tch(gdf, path_img, path_tabla):
    """Genera el plano general (filtrado) y el cuadro estadístico (sin filtrar) con colores coordinados."""
    gdf = gdf.copy()
    
    # NUEVA CATEGORIZACIÓN EXTENDIDA (Aplica para mapa y estadísticas)
    gdf['categoria'] = pd.cut(
        gdf['vryldrcane'], 
        bins=[-1, 40, 60, 80, 100, 9999],
        labels=['< 40 Tn/Ha', '40 a 60 Tn/Ha', '60 a 80 Tn/Ha', '80 a 100 Tn/Ha', '> 100 Tn/Ha'], 
        include_lowest=True
    )
    
    # --- 1. GENERAR PLANO VISUAL (Filtrado por distancia_real < 3) ---
    col_filtro = 'distancia_real' if 'distancia_real' in gdf.columns else 'distance'
    gdf_plano = gdf[gdf[col_filtro] < 3].copy()
    
    # --- ANTES: figsize=(10, 5) ---
    # --- AHORA: Subimos a (14, 7) para un lienzo base más grande ---
    fig, ax = plt.subplots(figsize=(14, 7))
    minx, miny, maxx, maxy = gdf_plano.total_bounds
    cx, cy = (minx + maxx) / 2, (miny + maxy) / 2
    rango = max(maxx - minx, (maxy - miny) * 2.5) * 1.2
    
    # Renderizar tramos por categoría de color con el grosor definido (linewidth)
    for cat, data in gdf_plano.groupby('categoria', observed=False):
        data.plot(ax=ax, color=colores_tch.get(cat, '#7f8c8d'), linewidth=0.2, label=str(cat))
        
    ax.set_xlim(cx - rango / 2, cx + rango / 2)
    ax.set_ylim(cy - (rango / 2.5) / 2, cy + (rango / 2.5) / 2)
    ax.axis('off')
    plt.title(f"PLANO GENERAL DE RENDIMIENTO TCH - PROPIEDAD", fontsize=12, fontweight='bold', pad=12)
    plt.legend(loc='lower right', frameon=True)
    
    # --- ANTES: dpi=100 ---
    # --- AHORA: Subimos a dpi=300 para calidad de impresión profesional (Alta Definición) ---
    plt.savefig(path_img, dpi=300, bbox_inches='tight')
    plt.close()

    # --- 2. GENERAR CUADRO ESTADÍSTICO (Sin Filtrar - Totalidad de Datos Reales) ---
    gdf['area_ha'] = (gdf['distance'] * gdf['swathwidth']) / 10000
    
    resumen = gdf.groupby('categoria', observed=False).agg(
        Metros=('distance', 'sum'),
        Hectareas=('area_ha', 'sum')
    ).reset_index()
    
    total_m = resumen['Metros'].sum() if resumen['Metros'].sum() > 0 else 1
    resumen['Distribución (%)'] = (resumen['Metros'] / total_m) * 100
    
    col_name = 'TCH (Sensor)'
    resumen.columns = [col_name, 'Longitud Total', 'Superficie', 'Distribución (%)']
    
    # Función de pintado dinámico de celdas según categoría (Sincronizado)
    def aplicar_color_categoria(row):
        categoria_fila = row[col_name]
        color_hex = colores_tch.get(categoria_fila, '#ffffff')
        estilos = [''] * len(row)
        idx_distribucion = row.index.get_loc('Distribución (%)')
        
        # El texto va en negro si el fondo es amarillo o naranja/verde claro para una lectura nítida
        text_color = 'black' if color_hex in ['#fdae61', '#ffffbf', '#a6d96a'] else 'white'
        estilos[idx_distribucion] = f'background-color: {color_hex}; color: {text_color}; font-weight: bold;'
        return estilos

    style = resumen.style.apply(aplicar_color_categoria, axis=1).hide(axis='index')
    style = style.format({
        'Longitud Total': '{:,.1f} m',
        'Superficie': '{:,.2f} ha',
        'Distribución (%)': '{:,.1f} %'
    })
    
    dfi.export(style, path_tabla, table_conversion='chrome')

# --- 3. FUNCIÓN DE RENDERIZADO PRINCIPAL ---
def generar_reporte_servicio_completo(propiedad, fecha_control):
    try:
        plantilla_path = os.path.join(ruta_plantilla, "tpl_inf_cosecha_servicio_F1.docx")
        doc = DocxTemplate(plantilla_path)
        
        anio_sel = fecha_control.year
        mes_nombre_sel = meses_espanol[fecha_control.month]
        semana_sel = fecha_control.isocalendar()[1]
        
        # --- BLOQUE 1: CÁLCULOS MATEMÁTICOS (df_ha) ---
        df_prop = df_ha[(df_ha['Propiedad'] == propiedad) & (df_ha['Año'] == anio_sel)]
        sem_area = df_prop[(df_prop['Mes'].str.lower() == mes_nombre_sel) & (df_prop['Semana'] == semana_sel)]['Area Cosechada'].sum()
        sem_tn = df_prop[(df_prop['Mes'].str.lower() == mes_nombre_sel) & (df_prop['Semana'] == semana_sel)]['Tn Real'].sum()
        sem_tch = (sem_tn / sem_area) if sem_area > 0 else 0.0
        
        mes_area = df_prop[df_prop['Mes'].str.lower() == mes_nombre_sel]['Area Cosechada'].sum()
        mes_tn = df_prop[df_prop['Mes'].str.lower() == mes_nombre_sel]['Tn Real'].sum()
        mes_tch = (mes_tn / mes_area) if mes_area > 0 else 0.0
        
        total_area = df_prop['Area Cosechada'].sum()
        total_tn = df_prop['Tn Real'].sum()
        total_tch = (total_tn / total_area) if total_area > 0 else 0.0
        
        # --- BLOQUE 2: PROCESAMIENTO DE LAS DOS TABLAS DE LOTES ---
        df_lotes_prop = df_detalles[df_detalles['Propiedad'] == propiedad].copy()
        
        df_lotes_prop['Area Cosecha'] = df_lotes_prop['Area Cosecha'].fillna(0.0)
        df_lotes_prop['Tn Real'] = df_lotes_prop['Tn Real'].fillna(0.0)
        df_lotes_prop['TCH Real'] = df_lotes_prop['TCH Real'].fillna(0.0)
        df_lotes_prop['Variedad'] = df_lotes_prop['Variedad'].fillna("S/V")
        df_lotes_prop['Soca'] = df_lotes_prop['Soca'].fillna(0)
        df_lotes_prop['POND ME'] = df_lotes_prop['POND ME'].fillna(0.0)
        df_lotes_prop['POND PCF'] = df_lotes_prop['POND PCF'].fillna(0.0)
        df_lotes_prop['POND FIBRA'] = df_lotes_prop['POND FIBRA'].fillna(0.0)
        df_lotes_prop['POND PUREZA'] = df_lotes_prop['POND PUREZA'].fillna(0.0)
        
        lista_lotes_jinja = []
        for _, fila in df_lotes_prop.iterrows():
            avance_original = fila['Avance']
            if pd.isna(avance_original):
                avance_final = "0 %"
            else:
                try:
                    avance_limpio = str(avance_original).replace('%', '').replace(',', '.').strip()
                    avance_num = float(avance_limpio)
                    if avance_num > 100.0: avance_num = 100.0
                    avance_final = f"{int(round(avance_num))} %"
                except:
                    avance_final = str(avance_original)
            
            lista_lotes_jinja.append({
                "lote": str(fila['Lote']), "estado": str(fila['Estado Cosecha']), "avance": avance_final,
                "variedad": str(fila['Variedad']), "soca": str(int(fila['Soca'])),
                "area": f"{fila['Area Cosecha']:,.2f}", "tn": f"{fila['Tn Real']:,.2f}", "tch": f"{fila['TCH Real']:,.2f}",
                "me": f"{fila['POND ME']:,.2f} %", "pcf": f"{fila['POND PCF']:,.2f}",
                "fibra": f"{fila['POND FIBRA']:,.2f} %", "pureza": f"{fila['POND PUREZA']:,.2f} %"
            })
            
        # --- BLOQUE 3: CONSTRUCCIÓN DEL MAPA Y LA TABLA GENERAL (PostgreSQL) ---
        img_temp_mapa = "tmp_tch_general.png"
        img_temp_tabla = "tmp_tch_general_tb.png"
        gdf_geo = obtener_telemetria_propiedad(propiedad)
        tiene_geo = not gdf_geo.empty
        
        if tiene_geo:
            generar_plano_y_tabla_tch(gdf_geo, img_temp_mapa, img_temp_tabla)
            
        # --- BLOQUE 4: CONTEXTO INTEGRADO PARA JINJA ---
        contexto_r = {
            "fecha": fecha_control.strftime("%d/%m/%Y"),
            "propiedad": propiedad,
            "num_semana": semana_sel,
            "nombre_mes": mes_nombre_sel.capitalize(),
            
            "sem_area": f"{sem_area:,.2f}", "sem_tn": f"{sem_tn:,.2f}", "sem_tch": f"{sem_tch:,.2f}",
            "mes_area": f"{mes_area:,.2f}", "mes_tn": f"{mes_tn:,.2f}", "mes_tch": f"{mes_tch:,.2f}",
            "total_area": f"{total_area:,.2f}", "total_tn": f"{total_tn:,.2f}", "total_tch": f"{total_tch:,.2f}",
            
            "lotes": lista_lotes_jinja,
            
            # Inyección de Imagen del Plano e Imagen del Cuadro Estadístico
            "img_plano_tch_general": InlineImage(doc, img_temp_mapa, width=Inches(6.0)) if tiene_geo else "[Sin datos geoespaciales]",
            "img_tabla_tch_general": InlineImage(doc, img_temp_tabla, width=Inches(4.5)) if tiene_geo else ""
        }
        
        doc.render({"r": contexto_r})
        nombre_archivo = f"Reporte_Servicio_Completo_{propiedad.replace(' ', '_')}_Semana_{semana_sel}.docx"
        ruta_salida = os.path.join(ruta_base, nombre_archivo)
        doc.save(ruta_salida)
        
        # Limpieza de archivos temporales
        for f in [img_temp_mapa, img_temp_tabla]:
            if os.path.exists(f): os.remove(f)
        
        return f"✅ ¡Reporte consolidado con éxito!\n📂 Archivo: {ruta_salida}\n🌍 Incluye: Tablas operativas, de calidad, plano y cuadro estadístico general de TCH de la propiedad."
        
    except Exception as e:
        return f"❌ Error al procesar el reporte: {str(e)}"

# --- 5. INTERFAZ GRÁFICA INTERACTIVA ---
combo_propiedad = widgets.Dropdown(options=sorted(df_ha['Propiedad'].unique()), description='Propiedad:')
control_fecha = widgets.DatePicker(description='Fecha Consulta:', value=datetime.now().date())
btn_generar = widgets.Button(description='Generar Reporte Completo', button_style='success', icon='map')
output_data = widgets.Output()

def controlar_interfaz(*args):
    with output_data:
        clear_output()
        if combo_propiedad.value and control_fecha.value:
            print(f"Propiedad: {combo_propiedad.value} | Fecha seleccionada: {control_fecha.value.strftime('%d/%m/%Y')}")
            cant_lotes = len(df_detalles[df_detalles['Propiedad'] == combo_propiedad.value])
            print(f"📋 Se procesará la cartografía unificada y el cuadro estadístico global para {cant_lotes} lotes.")
            display(btn_generar)

def al_hacer_clic_generar(b):
    with output_data:
        print("\n📥 Conectando a PostGIS, extrayendo telemetría general y renderizando tablas con Jinja...")
        mensaje = generar_reporte_servicio_completo(combo_propiedad.value, control_fecha.value)
        print(mensaje)

combo_propiedad.observe(controlar_interfaz, 'value')
control_fecha.observe(controlar_interfaz, 'value')
btn_generar.on_click(al_hacer_clic_generar)

controlar_interfaz()
display(widgets.HBox([combo_propiedad, control_fecha]), output_data)

Output()